In [2]:
from __future__ import print_function

import os
import gc
import torch
import random
import numpy as np
from loader.MSVD import MSVD
from loader.MSRVTT import MSRVTT
from run import build_loader, run
from config import TrainConfig as C
from tensorboardX import SummaryWriter
from models.abd_transformer import ABDTransformer
from torch.optim.lr_scheduler import ReduceLROnPlateau
from utils import evaluate, get_lr, load_checkpoint, save_checkpoint, test, train
import time

In [ ]:
def build_loaders():
    corpus = None
    if C.corpus == "MSVD":
        corpus = MSVD(C)
    elif C.corpus == "MSR-VTT":
        corpus = MSRVTT(C)
    print('#vocabs: {} ({}), #words: {} ({}). Trim words which appear less than {} times.'.format(
        corpus.vocab.n_vocabs, corpus.vocab.n_vocabs_untrimmed, corpus.vocab.n_words,
        corpus.vocab.n_words_untrimmed, C.loader.min_count))
    return corpus.train_data_loader, corpus.val_data_loader, corpus.test_data_loader, corpus.vocab

train_iter, val_iter, test_iter, vocab = build_loaders()

Enter the load4 method.  data_loader_fusion.py--row279


  7%|▋         | 85/1200 [00:01<00:24, 46.36it/s] 

In [ ]:
for batch in train_iter:
    break

In [57]:
def parse_batch(batch, feature_mode):
    if feature_mode == 'one':
        vids, feats, r2l_captions, l2r_captions = batch
        feats = [feat for feat in feats]
        feats = torch.cat(feats, dim=2)
        r2l_captions = r2l_captions.long()
        l2r_captions = l2r_captions.long()
        return vids, feats, r2l_captions, l2r_captions
    elif feature_mode == 'two':
        vids, image_feats, motion_feats, r2l_captions, l2r_captions = batch
        image_feats = [feat for feat in image_feats]
        motion_feats = [feat for feat in motion_feats]
        image_feats = torch.cat(image_feats, dim=2)
        motion_feats = torch.cat(motion_feats, dim=2)
        feats = (image_feats, motion_feats)
        r2l_captions = r2l_captions.long()
        l2r_captions = l2r_captions.long()
        return vids, feats, r2l_captions, l2r_captions
    elif feature_mode == 'three':
        vids, image_feats, motion_feats, object_feats, r2l_captions, l2r_captions = batch
        image_feats = [feat for feat in image_feats]
        motion_feats = [feat for feat in motion_feats]
        object_feats = [feat for feat in object_feats]
        image_feats = torch.cat(image_feats, dim=2)
        motion_feats = torch.cat(motion_feats, dim=2)
        object_feats = torch.cat(object_feats, dim=2)
        feats = (image_feats, motion_feats, object_feats)
        r2l_captions = r2l_captions.long()
        l2r_captions = l2r_captions.long()
        return vids, feats, r2l_captions, l2r_captions
    elif feature_mode == 'four':
        vids, image_feats, motion_feats, object_feats, rel_feats, r2l_captions, l2r_captions = batch
        print("========================== before feats ==========================")        
        image_feats = [feat for feat in image_feats]
        motion_feats = [feat for feat in motion_feats]
        object_feats = [feat for feat in object_feats]
        rel_feats = [feat for feat in rel_feats]
        
        assert len(image_feats) == len(motion_feats) == len(object_feats) == len(rel_feats) == 1
        
        print(f"image_feats shape: {image_feats[0].shape}")
        print(f"motion_feats shape: {motion_feats[0].shape}")
        print(f"object_feats shape: {object_feats[0].shape}")
        print(f"rel_feats shape: {rel_feats[0].shape}")
        # return image_feats
        image_feats = torch.cat(image_feats, dim=2)
        motion_feats = torch.cat(motion_feats, dim=2)
        object_feats = torch.cat(object_feats, dim=2)
        rel_feats = torch.cat(rel_feats, dim=2)
        
        print("========================== after feats ==========================")        
        print(f"image_feats shape: {image_feats.shape}")
        print(f"motion_feats shape: {motion_feats.shape}")
        print(f"object_feats shape: {object_feats.shape}")
        print(f"rel_feats shape: {rel_feats.shape}")
        
        feats = (image_feats, motion_feats, object_feats, rel_feats)
        r2l_captions = r2l_captions.long()
        l2r_captions = l2r_captions.long()
        # return vids, feats, r2l_captions, l2r_captions, my_image_feats
        return None

In [61]:
# _, feats, r2l_captions, l2r_captions = 
i = 5
for batch in test_iter:
    # break
    my_image_feats = parse_batch(batch, "four")
    print()
    i += 1
    if i == 10:
        break

========================== before feats ==========================
image_feats shape: torch.Size([1, 50, 1536])
motion_feats shape: torch.Size([1, 50, 1024])
object_feats shape: torch.Size([1, 50, 1028])
rel_feats shape: torch.Size([1, 50, 300])
========================== after feats ==========================
image_feats shape: torch.Size([1, 50, 1536])
motion_feats shape: torch.Size([1, 50, 1024])
object_feats shape: torch.Size([1, 50, 1028])
rel_feats shape: torch.Size([1, 50, 300])

========================== before feats ==========================
image_feats shape: torch.Size([1, 50, 1536])
motion_feats shape: torch.Size([1, 50, 1024])
object_feats shape: torch.Size([1, 50, 1028])
rel_feats shape: torch.Size([1, 50, 300])
========================== after feats ==========================
image_feats shape: torch.Size([1, 50, 1536])
motion_feats shape: torch.Size([1, 50, 1024])
object_feats shape: torch.Size([1, 50, 1028])
rel_feats shape: torch.Size([1, 50, 300])

================

In [53]:
my_image_feats

[tensor([[[0.4182, 0.6835, 0.4278,  ..., 0.5063, 0.1470, 0.0063],
          [0.6807, 0.8484, 0.3244,  ..., 1.1886, 0.4686, 0.0407],
          [0.6768, 0.7993, 0.2897,  ..., 1.1665, 0.4517, 0.0419],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]])]

In [56]:
# This line concatenates the features along the last dimension, 
# which is typically the feature dimension in video processing tasks.
torch.cat(my_image_feats, dim=0)

# assert two tensor array are equal
assert torch.equal(my_image_feats[0], torch.cat(my_image_feats, dim=0))

In [1]:
def build_model(vocab):
    try:
        model = ABDTransformer(vocab, C.feat.size, C.transformer.d_model, C.transformer.d_ff,
                               C.transformer.n_heads, C.transformer.n_layers, C.transformer.dropout,
                               C.feat.feature_mode,
                               n_heads_big=C.transformer.n_heads_big, select_num=C.transformer.select_num)
    except:
        model = ABDTransformer(vocab, C.feat.size, C.transformer.d_model, C.transformer.d_ff,
                               C.transformer.n_heads, C.transformer.n_layers, C.transformer.dropout,
                               C.feat.feature_mode,
                               n_heads_big=C.transformer.n_heads_big)
    
    return model